# Домашняя работа

Выполнили:

Мамиева Мария Алановна 424

Пугачев Егор Константинович 425

## 1. Машинки: поиск A*

Состояние — расположение машин на поле из 4 строк и 5 столбцов.
Оператор — сдвиг одной машины на одну клетку вверх, вниз, влево или вправо.

Цель (0 — свободная клетка):
```
2 3 0 8 9
2 3 0 8 9
1 1 5 7 7
1 1 4 6 6
```

Предложим начальное состояние, полученное из цели сдвигами машины 3 вправо и машины 2 вправо:
```
0 2 3 8 9
0 2 3 8 9
1 1 5 7 7
1 1 4 6 6
```

Используем A*: выбираем из OPEN состояние с наименьшим f = g + h.
Здесь g — число сделанных ходов, h — число машин не на целевых местах.
Считаем именно машины, а не занятые ими клетки. Один ход меняет положение только
одной машины, поэтому h не завышает оставшееся число ходов.
Раскрытое состояние переносим в CLOSED, его новых потомков добавляем в OPEN.
При совпадении с целью заканчиваем поиск.

1. Корень — начальная матрица, рядом оценка **0 + 2**.
2. Единственный допустимый ход — машина 2 влево. Получится матрица ниже, оценка **1 + 1**:
```
2 0 3 8 9
2 0 3 8 9
1 1 5 7 7
1 1 4 6 6
```
3. Из неё два хода: машина 2 вправо (возврат к корню, **2 + 2**, повтор не добавляем) и машина 3 влево (целевая матрица, **2 + 0**),вверх очевидно перчему нет, вниз невозможно, тк машинка снизу 2 на 2 и ника её не переместить когда '.' вертикальны.




## 2. Путь между страницами

Смоделируем сайт: создадим 8 адресов, для каждого случайно выберем две ссылки
на другие страницы. Граф ориентированный: ссылка A → B не означает B → A.
Храним его словарём «адрес: список ссылок». Это модель страниц, без скачивания HTML.

Ищем путь с помощью BFS: помещаем начальную страницу в очередь, затем по очереди
просматриваем её ссылки, запоминая, откуда пришли. Повторно страницу не добавляем.
BFS находит кратчайший путь по числу переходов. Случайный граф может не содержать пути.


In [ ]:
import random
from collections import deque

rng = random.Random(7)  # одинаковый граф при повторном запуске
urls = [f"https://site.test/page{i}" for i in range(1, 9)]
graph = {u: rng.sample([v for v in urls if v != u], 2) for u in urls}

for u, links in graph.items():
    print(u, "->", ", ".join(links))


In [ ]:
def bfs(graph, start, goal):
    queue = deque([start])
    parent = {start: None}
    while queue:
        u = queue.popleft()
        if u == goal:
            path = []
            while u is not None:
                path.append(u)
                u = parent[u]
            return path[::-1]
        for v in graph.get(u, []):
            if v not in parent:
                parent[v] = u
                queue.append(v)
    return None

path = bfs(graph, urls[0], urls[-1])
if path:
    print("Путь:", " → ".join(path))
    print("Число переходов:", len(path) - 1)
else:
    print("Пути нет")

assert bfs(graph, urls[0], urls[0]) == [urls[0]]
assert bfs({"a": [], "b": []}, "a", "b") is None


Для настоящих страниц вместо готового списка ссылок нужно скачивать HTML
и извлекать адреса из `<a href="...">`. Сам BFS остаётся тем же.
Двусторонний поиск требует ещё и входящих ссылок: по одному HTML целевой. В нашем готовом графе обратные связи можно построить,
но в жизни не уверен что это всегда возможно, ну точнее требует некого алгоритма который будет обходить и периодически собирать/обновлять информацию об обратных ссылках, но опять же он не найдёт прям все.

## 3. Коммивояжёр: генетический алгоритм

10 городов стоят по окружности. Рёбра между соседями имеют вес 1,
остальные — случайный вес от 8 до 25. Хромосома — перестановка городов,
начинающаяся с A; возвращение в A учитывается при подсчёте длины.

Использованы методы: случайное формирование начальной популяции (6 маршрутов), турнирный отбор (лучший из трёх), перестановка случайно выбранных битов местами и реверс битовой строки — здесь вместо битов используются города, а переворачивается случайный участок маршрута. Вероятности мутаций — 0,45 и 0,30, применяются независимо.

Скрещивание я адаптировал: сохраняем участок первого родителя, остальные города добавляем в порядке второго без повторений. Обычное двухточечное скрещивание может повторить города и потерять другие.

Две лучшие особи сохраняются, четыре создаются заново. После 120 поколений без улучшения два потомка заменяются случайными маршрутами. Оценка — длина замкнутого маршрута L, чем меньше, тем лучше. Остановка: L = 10 или 2500 поколений. Обход окружности оптимален: в цикле 10 рёбер, каждое весом не меньше 1.


In [ ]:
import matplotlib.pyplot as plt

import math
import random
import string

import numpy as np

N_CITIES = 10
CITY_NAMES = list(string.ascii_uppercase[:N_CITIES])
POPULATION_SIZE = 6
OPTIMUM = N_CITIES


def make_distance_matrix(n=10, seed=2026):
    rng = random.Random(seed)
    matrix = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            are_ring_neighbours = (j == i + 1) or (i == 0 and j == n - 1)
            weight = 1 if are_ring_neighbours else rng.randint(8, 25)
            matrix[i][j] = matrix[j][i] = weight
    return matrix


DISTANCES = make_distance_matrix(N_CITIES)


def route_length(route, distances=DISTANCES):
    return sum(
        distances[route[i]][route[(i + 1) % len(route)]]
        for i in range(len(route))
    )


def route_text(route):
    return "-".join(CITY_NAMES[i] for i in route) + "-A"


def random_route(rng):
    tail = list(range(1, N_CITIES))
    rng.shuffle(tail)
    return tuple([0] + tail)


def tournament_selection(population, rng, size=3):
    competitors = rng.sample(population, k=min(size, len(population)))
    return min(competitors, key=route_length)


def ordered_crossover(parent1, parent2, rng):
    first, second = parent1[1:], parent2[1:]
    left, right = sorted(rng.sample(range(len(first)), 2))
    right += 1

    child = [None] * len(first)
    child[left:right] = first[left:right]
    remaining = [gene for gene in second if gene not in child]
    free_positions = [i for i, gene in enumerate(child) if gene is None]
    for position, gene in zip(free_positions, remaining):
        child[position] = gene
    return tuple([0] + child)


def mutate(route, rng, swap_probability=0.45, inversion_probability=0.30):
    genes = list(route)
    if rng.random() < swap_probability:
        i, j = rng.sample(range(1, N_CITIES), 2)
        genes[i], genes[j] = genes[j], genes[i]
    if rng.random() < inversion_probability:
        i, j = sorted(rng.sample(range(1, N_CITIES), 2))
        genes[i:j + 1] = reversed(genes[i:j + 1])
    return tuple(genes)


def genetic_algorithm(seed=41, max_generations=2500):
    rng = random.Random(seed)
    population = [random_route(rng) for _ in range(POPULATION_SIZE)]
    history = []
    stagnation = 0
    previous_best = float("inf")

    for generation in range(max_generations + 1):
        population = sorted(population, key=route_length)
        best_length = route_length(population[0])
        history.append({
            "generation": generation,
            "population": tuple(population),
            "best_route": population[0],
            "best_length": best_length,
            "mean_length": sum(map(route_length, population)) / len(population),
        })

        if best_length == OPTIMUM:
            break

        stagnation = stagnation + 1 if best_length == previous_best else 0
        previous_best = best_length

        new_population = population[:2]
        while len(new_population) < POPULATION_SIZE:
            parent1 = tournament_selection(population, rng)
            parent2 = tournament_selection(population, rng)
            child = ordered_crossover(parent1, parent2, rng)
            child = mutate(child, rng)
            new_population.append(child)

        if stagnation >= 120:
            new_population[-2:] = [random_route(rng), random_route(rng)]
            stagnation = 0

        population = new_population

    return history


history = genetic_algorithm()
final = history[-1]

print(f"Поколений выполнено: {final['generation']}")
print(f"Лучший маршрут: {route_text(final['best_route'])}")
print(f"Длина: {final['best_length']}")
print(f"Глобальный оптимум найден: {final['best_length'] == OPTIMUM}")
assert sorted(final["best_route"]) == list(range(N_CITIES))


In [ ]:

def plot_route(ax, route, title):
    angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, N_CITIES, endpoint=False)
    points = np.column_stack([np.cos(angles), np.sin(angles)])

    # Полный граф — тонкими серыми линиями.
    for i in range(N_CITIES):
        for j in range(i + 1, N_CITIES):
            ax.plot(points[[i, j], 0], points[[i, j], 1],
                    color="#D9DEE7", linewidth=0.55, zorder=0)

    # Оптимальные рёбра окружности — зелёным.
    for i in range(N_CITIES):
        j = (i + 1) % N_CITIES
        ax.plot(points[[i, j], 0], points[[i, j], 1],
                color="#2A9D8F", linewidth=2.2, zorder=1)

    # Текущий лучший маршрут — оранжевым.
    closed_route = list(route) + [route[0]]
    for i, j in zip(closed_route, closed_route[1:]):
        ax.plot(points[[i, j], 0], points[[i, j], 1],
                color="#F4A261", linewidth=3.2, alpha=0.9, zorder=2)

    palette = plt.cm.tab10(np.linspace(0, 1, N_CITIES))
    ax.scatter(points[:, 0], points[:, 1], s=560, c=palette,
               edgecolor="white", linewidth=2, zorder=3)
    for i, (x, y) in enumerate(points):
        ax.text(x, y, CITY_NAMES[i], ha="center", va="center",
                color="white", fontweight="bold", fontsize=12, zorder=4)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")


snapshot_indices = sorted(set([0, len(history) // 2, len(history) - 1]))
fig, axes = plt.subplots(1, len(snapshot_indices), figsize=(15, 4.8), constrained_layout=True)
if len(snapshot_indices) == 1:
    axes = [axes]
for ax, index in zip(axes, snapshot_indices):
    item = history[index]
    plot_route(
        ax,
        item["best_route"],
        f"Поколение {item['generation']}\nL = {item['best_length']}",
    )
plt.suptitle("Как лучший маршрут превращается в обход по окружности",
             fontsize=15, fontweight="bold")
plt.show()

generations = [item["generation"] for item in history]
best_values = [item["best_length"] for item in history]
mean_values = [item["mean_length"] for item in history]

plt.figure(figsize=(11, 4.5))
plt.plot(generations, mean_values, color="#577590", alpha=0.65,
         label="средняя длина в популяции")
plt.plot(generations, best_values, color="#E76F51", linewidth=2.5,
         label="лучшая длина")
plt.axhline(OPTIMUM, color="#2A9D8F", linestyle="--", label="оптимум L=10")
plt.xlabel("Поколение")
plt.ylabel("Длина маршрута")
plt.title("Сходимость генетического алгоритма")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


In [ ]:

print("Финальная популяция (6 хромосом):")
for number, route in enumerate(final["population"], start=1):
    print(f"{number}: {route_text(route):24s}  L={route_length(route)}")


В этом запуске получился обход по окружности. При другом seed результат
и число поколений могут измениться: генетический алгоритм не гарантирует
достижение оптимума за фиксированное время.